# MyTravels — Preview Runbook

Runs the full MyTravels stack via Docker Compose. No local SDK or build tools required.

| Service | Purpose | Port(s) |
|---------|---------|--------|
| PostgreSQL | Primary database | 5432 |
| RabbitMQ | Message broker | 5672 · 15672 (UI) |
| MinIO | Object storage | 9000 (API) · 9090 (Console) |

> Run each cell in order.

## Summary

This runbook starts the MyTravels **infrastructure services** locally using Docker Compose — PostgreSQL, RabbitMQ, and MinIO — plus the database migration jobs. The .NET application services (API and Messaging) are run separately from this starter project. It walks through:

1. **Copy environment file** — copy `.env.example` to `.env` if one doesn't already exist.
2. **Load environment variables** — read the `.env` file in this directory into the notebook's environment.
3. **Start the stack** — bring up PostgreSQL (port 5432), RabbitMQ (5672 / UI 15672), and MinIO (9000 / Console 9090) with `docker compose up -d`.
4. **Check service health** — verify all containers are running with `docker compose ps`.
5. **View logs** — tail recent logs for each service, including the DB migration jobs.
6. **Teardown** — stop everything and wipe volumes with `docker compose down --volumes`.

**Prerequisites:** Rancher Desktop (running), JupyterLab, and a `.env` file in this directory.

## Prerequisites

Rancher Desktop and JupyterLab install notes have moved to the root tool guides: [macOS](<../1-required tools (macos).md>) · [Ubuntu](<../1-required tools (ubuntu).md>) · [Windows](<../1-required tools (windows).md>).

Once Rancher Desktop is installed, open it and ensure the container engine is running before continuing.

- A `.env` file must exist in this directory (copy from `.env.example` if one exists)

---
## 1. Copy environment file

Copy `.env.example` to `.env` if `.env` doesn't already exist.

In [ ]:
%%bash
if [ -f .env ]; then
  echo ".env already exists, skipping."
else
  cp .env.example .env
  echo "Copied .env.example to .env"
fi

---
## 2. Load environment variables

In [ ]:
from pathlib import Path
import os

for line in Path(".env").read_text().splitlines():
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        key, _, value = line.partition("=")
        os.environ[key.strip()] = value.strip()

print("Loaded environment from .env")

---
## 3. Start the stack

In [ ]:
%%bash
docker compose up -d
echo "Stack started."

---
## 4. Check service health

In [ ]:
%%bash
docker compose ps

**Service UIs:**
- RabbitMQ Management: http://localhost:15672
- MinIO Console: http://localhost:9090

---
## 5. View logs (optional)

In [ ]:
%%bash
echo "=== minio ===" && docker compose logs --tail=20 minio
echo "=== rabbitmq ===" && docker compose logs --tail=20 rabbitmq
echo "=== postgres ===" && docker compose logs --tail=20 postgres
echo "=== cleanup-migrations ===" && docker compose logs --tail=20 cleanup-migrations
echo "=== migrate-core-db ===" && docker compose logs --tail=20 migrate-core-db

## 6. Teardown

In [ ]:
%%bash
# Stop and remove containers AND volumes — wipes all data
docker compose down --volumes
echo "Containers and volumes removed."